# Notebook 2.5: Schrödinger Cat States and Phase Space Interference
### Module 2: Phase Space and Quasiprobability Distributions
**Computational Quantum Optics Curriculum**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rfandan/computational-quantum-optics/blob/main/module_2_phase_space_quasiprobabilities/06_schrodinger_cat_states_phase_space.ipynb)

---

### Prerequisites
- Notebook 2.1: The Wigner Quasiprobability Distribution Fundamentals
- Notebook 2.2: Wigner Negativity as a Metric for Quantum Non-Classicality
- Notebook 1.2: Coherent States, Displacement, and Poissonian Statistics

### Pedagogical Learning Objectives
1. **Mathematical Structure of Optical Cat States:** Construct Even ($+$), Odd ($-$), and Yurke-Stoler (phase-shifted) macroscopic superpositions of coherent states: $|\psi_\pm\rangle = \mathcal{N}_\pm (|\alpha\rangle \pm |-\alpha\rangle)$.
2. **Phase-Space Interference Fringes:** Derive the exact analytical Wigner function $W_\pm(x, p) = \mathcal{N}_\pm^2 [W_\alpha + W_{-\alpha} \pm 2 W_{\text{int}}]$, proving how the central interference fringes oscillate as $\cos(2\sqrt{2}\alpha p)$ with deep negative lobes.
3. **Sub-Planck Phase-Space Structure (Zurek, 2001):** Understand why the fringe oscillation period $\Delta p \sim 1/\alpha$ scales below Planck's constant, providing Heisenberg-limited phase sensitivity $\Delta\theta \sim 1/(2|\alpha|)$.
4. **Decoherence Dynamics:** Model environmental photon loss, demonstrating why quantum interference fringes vanish at the accelerated rate $\Gamma_{\text{dec}} = 2\gamma |\alpha|^2$, turning a macroscopic quantum superposition into an incoherent statistical mixture.
5. **Numerical Simulation & QuTiP Verification:** Construct Fock-space representations, benchmark parity eigenvalues $\langle (-1)^{\hat{n}} \rangle = \pm 1$, and simulate Lindblad decoherence in phase space.


In [ ]:
# Setup, environment detection, and library imports
import os
import sys
import math
import numpy as np
import scipy.linalg as la
from scipy.special import genlaguerre
import matplotlib.pyplot as plt

# Ensure matplotlib handles headless environments gracefully
if 'MPLBACKEND' not in os.environ:
    os.environ['MPLBACKEND'] = 'Agg' if not hasattr(sys, 'ps1') else 'module://matplotlib_inline.backend_inline'

# Plot styling configuration for high-DPI publication graphics
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['lines.linewidth'] = 2.0
plt.rcParams['grid.alpha'] = 0.35

# Detect QuTiP availability with clean fallback
try:
    import qutip as qt
    HAS_QUTIP = True
    print(f"QuTiP {qt.__version__} successfully imported.")
except ImportError:
    HAS_QUTIP = False
    print("QuTiP not found. Using native NumPy/SciPy implementations.")

# Detect interactive widgets environment
try:
    from ipywidgets import interact, FloatSlider, Dropdown
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not found. Interactive exploration will run in static fallback mode.")


## 1. Mathematical Architecture of Optical Cat States

In 1935, Erwin Schrödinger conceived his famous thought experiment: a macroscopic cat entangles with a radioactive atom, placing it in a simultaneous superposition of "alive" and "dead".

In quantum optics, a single-mode coherent state $|\alpha\rangle$ represents a classical-like macroscopic wave packet with mean photon number $\bar{n} = |\alpha|^2$. An **optical Schrödinger cat state** is a quantum superposition of two macroscopically distinct, diametrically opposed coherent states $|\alpha\rangle$ and $|-\alpha\rangle$:

---

### 1. The Even Cat State ($+$ Parity)
$$
|\psi_+\rangle = \mathcal{N}_+ \left( |\alpha\rangle + |-\alpha\rangle \right), \qquad \mathcal{N}_+ = \frac{1}{\sqrt{2\left(1 + e^{-2|\alpha|^2}\right)}}
$$
In the Fock basis, all odd photon components cancel identically ($1 + (-1)^n = 0$ for odd $n$):
$$
|\psi_+\rangle = 2\mathcal{N}_+ e^{-|\alpha|^2/2} \sum_{m=0}^\infty \frac{\alpha^{2m}}{\sqrt{(2m)!}} |2m\rangle
$$
The even cat state is an **eigenstate of photon parity** with eigenvalue $+1$: $\langle (-1)^{\hat{n}} \rangle = +1$.

---

### 2. The Odd Cat State ($-$ Parity)
$$
|\psi_-\rangle = \mathcal{N}_- \left( |\alpha\rangle - |-\alpha\rangle \right), \qquad \mathcal{N}_- = \frac{1}{\sqrt{2\left(1 - e^{-2|\alpha|^2}\right)}}
$$
Contains **exclusively odd Fock states**:
$$
|\psi_-\rangle = 2\mathcal{N}_- e^{-|\alpha|^2/2} \sum_{m=0}^\infty \frac{\alpha^{2m+1}}{\sqrt{(2m+1)!}} |2m+1\rangle
$$
It has photon parity $-1$: $\langle (-1)^{\hat{n}} \rangle = -1$.

---

### 3. The Yurke-Stoler Cat State
$$
|\psi_{\text{YS}}\rangle = \frac{1}{\sqrt{2}} \left( |\alpha\rangle + i |-\alpha\rangle \right)
$$
Generated naturally by propagating a coherent state through a Kerr non-linear medium ($\hat{H}_{\text{Kerr}} = \hbar \chi (\hat{a}^\dagger)^2 \hat{a}^2$) at time $t = \pi / (2\chi)$ (*Phys. Rev. Lett.* 57, 13, 1986).


## 2. Phase-Space Interference: The Wigner Function of Cat States

Evaluating the Wigner quasiprobability distribution for the density matrix $\hat{\rho}_+ = |\psi_+\rangle\langle\psi_+|$ yields three distinct physical contributions:

$$
W_+(x, p) = \mathcal{N}_+^2 \Big[ \underbrace{W_{|\alpha\rangle}(x, p)}_{\text{"Alive" Wavepacket}} + \underbrace{W_{|-\alpha\rangle}(x, p)}_{\text{"Dead" Wavepacket}} + \underbrace{2 W_{\text{int}}(x, p)}_{\text{Quantum Coherence / Interference}} \Big]
$$

Assuming real displacement amplitude $\alpha > 0$:

1. **The Classical Gaussian Lobes:**
   Two positive Gaussian peaks centered at $(+\sqrt{2}\alpha, 0)$ and $(-\sqrt{2}\alpha, 0)$:
   $$
   W_{|\pm\alpha\rangle}(x, p) = \frac{1}{\pi} \exp\left[ -(x \mp \sqrt{2}\alpha)^2 - p^2 \right]
   $$
2. **The Quantum Interference Term:**
   Located exactly midway at the phase-space origin $(x = 0, p = 0)$:
   $$
   W_{\text{int}}(x, p) = \frac{2}{\pi} \exp\left[ -(x^2 + p^2) \right] \cos\left( 2\sqrt{2}\alpha \, p \right)
   $$

For the **odd cat state** $|\psi_-\rangle$, the interference term acquires a minus sign:
$$
W_{\text{int}}^-(x, p) = -\frac{2}{\pi} \exp\left[ -(x^2 + p^2) \right] \cos\left( 2\sqrt{2}\alpha \, p \right)
$$

---

### Origin Parity Theorem Verification:
At the phase-space origin $(x=0, p=0)$:
- For the Even Cat: $W_+(0, 0) = \frac{1}{\pi} \langle (-1)^{\hat{n}} \rangle = +\frac{1}{\pi} > 0$ (Constructive interference peak!).
- For the Odd Cat: $W_-(0, 0) = \frac{1}{\pi} \langle (-1)^{\hat{n}} \rangle = -\frac{1}{\pi} < 0$ (Destructive interference negative trough!).

---

### Sub-Planck Structure (Zurek, 2001)

Notice the spatial frequency of the interference cosine: $\cos(2\sqrt{2}\alpha \, p)$.
The fringe period along the momentum coordinate is:
$$
\Delta p_{\text{fringe}} = \frac{2\pi}{2\sqrt{2}\alpha} = \frac{\pi}{\sqrt{2}\alpha}
$$
As the macroscopic separation $2\alpha$ increases, the fringes pack together with an effective phase-space area:
$$
A_{\text{fringe}} \sim \Delta x \cdot \Delta p_{\text{fringe}} \sim \frac{1}{\alpha} \ll 1
$$
As shown by **Wojciech Zurek** (*Nature* 412, 712, 2001), these **sub-Planck structures** allow optical cat states to detect tiny phase rotations $\Delta\theta \sim 1/(2\alpha)$ with sensitivity far exceeding the classical Standard Quantum Limit!


In [ ]:
# Numerical Construction and Wigner Synthesis of Optical Cat States

def cat_state(alpha, cat_type='even', N=40):
    """Constructs Fock-basis state vector for Even, Odd, or Yurke-Stoler cat states."""
    a = np.diag(np.sqrt(np.arange(1, N)), k=1)
    vac = np.zeros(N, dtype=np.complex128)
    vac[0] = 1.0
    
    # Displacement operators D(+alpha) and D(-alpha)
    D_plus = la.expm(alpha * a.T.conj() - np.conj(alpha) * a)
    D_minus = la.expm(-alpha * a.T.conj() + np.conj(alpha) * a)
    
    psi_plus = D_plus @ vac
    psi_minus = D_minus @ vac
    
    if cat_type == 'even':
        psi = psi_plus + psi_minus
    elif cat_type == 'odd':
        psi = psi_plus - psi_minus
    elif cat_type == 'yurke-stoler':
        psi = psi_plus + 1j * psi_minus
    else:
        raise ValueError(f"Unknown cat type: {cat_type}")
        
    norm = np.linalg.norm(psi)
    psi = psi / norm
    
    # Calculate photon parity <(-1)^n>
    parity_op = np.diag([(-1.0)**n for n in range(N)])
    parity_val = np.real(np.vdot(psi, parity_op @ psi))
    
    return psi, parity_val

def compute_wigner_cat(psi, xvec, pvec):
    """Computes Wigner function of cat state on 2D grid via Fock-basis Laguerre expansion."""
    N = len(psi)
    X, P = np.meshgrid(xvec, pvec)
    alpha_grid = (X + 1j * P) / np.sqrt(2.0)
    z = 4.0 * np.abs(alpha_grid)**2
    exp_factor = (1.0 / np.pi) * np.exp(-2.0 * np.abs(alpha_grid)**2)
    
    fact = np.array([float(math.factorial(n)) for n in range(N)])
    rho = np.outer(psi, psi.conj())
    
    W = np.zeros_like(X, dtype=np.complex128)
    for n in range(N):
        if abs(rho[n, n]) > 1e-15:
            W += rho[n, n] * ((-1)**n) * genlaguerre(n, 0)(z)
        for m in range(n + 1, N):
            if abs(rho[n, m]) > 1e-15:
                d = m - n
                coeff = ((-1)**n) * np.sqrt(fact[n] / fact[m]) * ((2.0 * np.conj(alpha_grid))**d)
                term = coeff * genlaguerre(n, d)(z)
                W += rho[n, m] * term + rho[m, n] * np.conj(term)
                
    return np.real(W) * exp_factor

# Benchmark: Even vs. Odd Cat States for alpha = 2.0
alpha_cat = 2.0
N_cat = 45

psi_even, parity_even = cat_state(alpha_cat, 'even', N_cat)
psi_odd, parity_odd = cat_state(alpha_cat, 'odd', N_cat)
psi_ys, parity_ys = cat_state(alpha_cat, 'yurke-stoler', N_cat)

x_cat = np.linspace(-4.5, 4.5, 121)
p_cat = np.linspace(-4.5, 4.5, 121)

W_even = compute_wigner_cat(psi_even, x_cat, p_cat)
W_odd = compute_wigner_cat(psi_odd, x_cat, p_cat)

center_idx = len(x_cat) // 2
print("--- Cat State Parity and Origin Benchmarking ---")
print(f"Even Cat Parity <(-1)^n>: {parity_even:+.6f} | Expected: +1.000000")
print(f"Even Cat W(0, 0):         {W_even[center_idx, center_idx]:+.6f} | Expected: {1.0/np.pi:+.6f}")
print(f"Odd Cat Parity <(-1)^n>:  {parity_odd:+.6f} | Expected: -1.000000")
print(f"Odd Cat W(0, 0):          {W_odd[center_idx, center_idx]:+.6f} | Expected: {-1.0/np.pi:+.6f} (Negative!)")

assert abs(parity_even - 1.0) < 1e-6, "Even cat parity failed!"
assert abs(parity_odd - (-1.0)) < 1e-6, "Odd cat parity failed!"
assert abs(W_even[center_idx, center_idx] - 1.0/np.pi) < 1e-3, "Even cat W(0,0) failed!"
assert abs(W_odd[center_idx, center_idx] - (-1.0/np.pi)) < 1e-3, "Odd cat W(0,0) failed!"
print("Analytical parity and Wigner origin values perfectly verified!")


In [ ]:
# 3-Panel Publication Gallery: Even, Odd, and Yurke-Stoler Cat States

W_ys = compute_wigner_cat(psi_ys, x_cat, p_cat)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5))

# Universal color limits
w_lim = 0.35
levels_cat = np.linspace(-w_lim, w_lim, 45)

# 1. Even Cat State
cf1 = ax1.contourf(x_cat, p_cat, W_even, levels=levels_cat, cmap='RdBu_r', extend='both')
ax1.contour(x_cat, p_cat, W_even, levels=[0.0], colors='black', linewidths=1.8, linestyles='--')
ax1.set_title(rf"Even Cat $|\psi_+\rangle$ ($\alpha={alpha_cat}$, $W(0)=+1/\pi$)")
plt.colorbar(cf1, ax=ax1, fraction=0.046, pad=0.04)

# 2. Odd Cat State
cf2 = ax2.contourf(x_cat, p_cat, W_odd, levels=levels_cat, cmap='RdBu_r', extend='both')
ax2.contour(x_cat, p_cat, W_odd, levels=[0.0], colors='black', linewidths=1.8, linestyles='--')
ax2.set_title(rf"Odd Cat $|\psi_-\rangle$ ($\alpha={alpha_cat}$, $W(0)=-1/\pi$)")
plt.colorbar(cf2, ax=ax2, fraction=0.046, pad=0.04)

# 3. Yurke-Stoler Cat State
cf3 = ax3.contourf(x_cat, p_cat, W_ys, levels=levels_cat, cmap='RdBu_r', extend='both')
ax3.contour(x_cat, p_cat, W_ys, levels=[0.0], colors='black', linewidths=1.8, linestyles='--')
ax3.set_title(rf"Yurke-Stoler $|\psi_{{\mathrm{{YS}}}}\rangle$ ($|\alpha\rangle + i|-\alpha\rangle$)")
plt.colorbar(cf3, ax=ax3, fraction=0.046, pad=0.04)

for ax in [ax1, ax2, ax3]:
    ax.set_aspect('equal')
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_xlabel(r"Quadrature $x$")
    ax.set_ylabel(r"Quadrature $p$")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 3. Environmental Decoherence: The Fleeting Life of Schrödinger's Cat

Why are macroscopic superpositions notoriously fragile and absent from our daily experience?

When an optical cat state interacts with an open dissipative environment (such as mirror transmission loss or fiber attenuation at rate $\gamma$), the system evolves under the **Lindblad master equation**:

$$
\frac{d\hat{\rho}}{dt} = \gamma \mathcal{D}[\hat{a}]\hat{\rho} = \gamma \left( \hat{a}\hat{\rho}\hat{a}^\dagger - \frac{1}{2}\hat{a}^\dagger\hat{a}\hat{\rho} - \frac{1}{2}\hat{\rho}\hat{a}^\dagger\hat{a} \right)
$$

---

### Analytical Solution for the Density Matrix under Decoherence

For an initial cat state $\hat{\rho}(0) = \mathcal{N}_+^2 (|\alpha\rangle + |-\alpha\rangle)(\langle\alpha| + \langle-\alpha|)$, the time-dependent density matrix decomposes into:

$$
\hat{\rho}(t) = \mathcal{N}_+^2 \left[ |\alpha(t)\rangle\langle\alpha(t)| + |-\alpha(t)\rangle\langle-\alpha(t)| + e^{-\Gamma_{\text{dec}} t} |\alpha(t)\rangle\langle-\alpha(t)| + e^{-\Gamma_{\text{dec}} t} |-\alpha(t)\rangle\langle\alpha(t)| \right]
$$

where:
1. **Energy Dissipation:** The amplitude of the coherent components decays slowly:
   $$
   \alpha(t) = \alpha(0) e^{-\gamma t / 2}
   $$
2. **Accelerated Decoherence Rate:** The off-diagonal quantum coherence terms decay at the **catastrophically amplified rate**:
   $$
   \Gamma_{\text{dec}} = 2\gamma |\alpha|^2 = 2\gamma \bar{n}
   $$

---

### The Macroscopic Scaling Disaster:
For a mesoscopic cat with $\bar{n} = 100$ photons:
$$
\Gamma_{\text{dec}} = 200 \, \gamma
$$
Quantum coherence (the Wigner interference fringes) is obliterated **200 times faster** than energy is lost! The system collapses into an ordinary classical statistical mixture:
$$
\hat{\rho}_{\text{mixed}} = \frac{1}{2} |\alpha(t)\rangle\langle\alpha(t)| + \frac{1}{2} |-\alpha(t)\rangle\langle-\alpha(t)|
$$
with zero Wigner negativity and zero quantum interference!


In [ ]:
# Simulating Decoherence: The Disappearance of Wigner Negativity under Photon Loss

def decohere_cat(alpha0, loss_factor, N=35):
    """Computes density matrix of cat state after loss factor eta = exp(-gamma * t)."""
    # loss_factor eta in [0, 1] (eta = 1: no loss, eta = 0: total loss)
    # alpha(t) = alpha0 * sqrt(eta)
    alpha_t = alpha0 * np.sqrt(loss_factor)
    
    # Coherence attenuation factor: exp(-2 * |alpha0|^2 * (1 - eta))
    decay_coh = np.exp(-2.0 * (alpha0**2) * (1.0 - loss_factor))
    
    a = np.diag(np.sqrt(np.arange(1, N)), k=1)
    vac = np.zeros(N, dtype=np.complex128); vac[0] = 1.0
    
    D_p = la.expm(alpha_t * a.T.conj() - alpha_t * a)
    D_m = la.expm(-alpha_t * a.T.conj() + alpha_t * a)
    
    psi_p = D_p @ vac
    psi_m = D_m @ vac
    
    # Construct mixed density matrix:
    # rho(t) = N^2 [ |+><+| + |-><-| + decay * (|+><-| + |-><+|) ]
    rho_pp = np.outer(psi_p, psi_p.conj())
    rho_mm = np.outer(psi_m, psi_m.conj())
    rho_pm = np.outer(psi_p, psi_m.conj())
    rho_mp = np.outer(psi_m, psi_p.conj())
    
    N_norm2 = 1.0 / (2.0 * (1.0 + np.exp(-2.0 * alpha0**2)))
    rho_t = N_norm2 * (rho_pp + rho_mm + decay_coh * (rho_pm + rho_mp))
    rho_t = rho_t / np.real(np.trace(rho_t))
    
    return rho_t, decay_coh

# Simulate 3 stages of decoherence: eta = 1.0 (pure), eta = 0.95 (slight loss), eta = 0.80 (decohered)
eta_stages = [1.0, 0.95, 0.80]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, eta_val in enumerate(eta_stages):
    rho_stage, coh_val = decohere_cat(alpha_cat, eta_val, N_cat)
    W_stage = compute_wigner_cat(rho_stage, x_cat, p_cat)
    
    min_w_val = np.min(W_stage)
    cf = axes[idx].contourf(x_cat, p_cat, W_stage, levels=levels_cat, cmap='RdBu_r', extend='both')
    axes[idx].contour(x_cat, p_cat, W_stage, levels=[0.0], colors='black', linewidths=1.8, linestyles='--')
    axes[idx].set_aspect('equal')
    axes[idx].set_xlim(-4, 4)
    axes[idx].set_ylim(-4, 4)
    axes[idx].set_xlabel("x")
    axes[idx].set_ylabel("p")
    axes[idx].set_title(rf"Transmission $\eta={eta_val:.2f}$ (Coh={coh_val:.2f}, Min $W={min_w_val:+.3f}$)")
    plt.colorbar(cf, ax=axes[idx], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

print("Decoherence simulation complete: rapid washout of central Wigner interference ripples confirmed!")


> **Physical Insight: Sub-Planck Compass States and Quantum Metrology**
> Why does nature embed interference fringes on scales smaller than $\hbar$ in phase space?
>
> In 2001, Wojciech Zurek proved that the sensitivity of a quantum state to external environmental perturbations or phase shifts is determined not by the size of the wavepackets $\Delta x$, but by the **spatial frequency of its quantum interference fringes**:
> $$
> \Delta p_{\text{fringe}} \sim \frac{\hbar}{2\alpha}
> $$
> A four-component cat state (the **"compass state"** $|\alpha\rangle + |-\alpha\rangle + |i\alpha\rangle + |-i\alpha\rangle$) creates a checkerboard of sub-Planck tiles in both $x$ and $p$. These states enable the most sensitive optical sensors known to physics, capable of detecting phase rotations at the fundamental **Heisenberg limit** ($\Delta\theta \sim 1/\bar{n}$) rather than the classical shot-noise limit ($\Delta\theta \sim 1/\sqrt{\bar{n}}$)!


In [ ]:
# Interactive Exploration: Real-Time Cat State & Decoherence Tuner

def explore_cat_state(alpha_val=2.0, loss_percent=0.0, cat_type='even'):
    """Interactive visualization of cat state phase space and environmental loss."""
    eta_val = 1.0 - (loss_percent / 100.0)
    N_calc = 40
    
    if loss_percent == 0.0:
        psi, _ = cat_state(alpha_val, cat_type, N_calc)
        W_map = compute_wigner_cat(psi, x_cat, p_cat)
    else:
        rho_t, _ = decohere_cat(alpha_val, eta_val, N_calc)
        W_map = compute_wigner_cat(rho_t, x_cat, p_cat)
        
    min_w = np.min(W_map)
    max_w = np.max(W_map)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    
    # 1. 2D Contour
    v_lim = max(0.35, max_w)
    cf = ax1.contourf(x_cat, p_cat, W_map, levels=np.linspace(-v_lim, v_lim, 40), cmap='RdBu_r', extend='both')
    ax1.contour(x_cat, p_cat, W_map, levels=[0.0], colors='black', linewidths=1.8, linestyles='--')
    ax1.set_aspect('equal')
    ax1.set_xlim(-4, 4)
    ax1.set_ylim(-4, 4)
    ax1.set_xlabel("x")
    ax1.set_ylabel("p")
    ax1.set_title(rf"{cat_type.capitalize()} Cat ($\alpha={alpha_val:.1f}$, Loss={loss_percent:.0f}%)")
    plt.colorbar(cf, ax=ax1, fraction=0.046, pad=0.04)
    
    # 2. 1D Cross-Section along p (Interference Ripples)
    mid_x_idx = len(x_cat) // 2
    slice_p = W_map[:, mid_x_idx]
    ax2.plot(p_cat, slice_p, 'r-', lw=2.2, label=r'Cross-section $W(0, p)$')
    ax2.axhline(0.0, color='black', lw=1.2, ls='--')
    ax2.fill_between(p_cat, slice_p, 0, where=(slice_p < 0), color='blue', alpha=0.3, label='Negative Interference')
    ax2.set_xlim(-4, 4)
    ax2.set_ylim(-0.4, 0.4)
    ax2.set_xlabel("p")
    ax2.set_ylabel("Amplitude")
    ax2.set_title(rf"Interference Fringes along Momentum: Min $W={min_w:+.3f}$")
    ax2.grid(True)
    ax2.legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()

# Launch interactive exploration
if HAS_WIDGETS:
    interact(explore_cat_state,
             alpha_val=FloatSlider(min=1.0, max=3.0, step=0.2, value=2.0, description=r'\alpha:'),
             loss_percent=FloatSlider(min=0.0, max=25.0, step=2.5, value=0.0, description='Loss %:'),
             cat_type=Dropdown(options=['even', 'odd', 'yurke-stoler'], value='even'))
else:
    # Static execution fallback for headless environments
    explore_cat_state(alpha_val=2.0, loss_percent=0.0, cat_type='even')


## 4. Self-Check Exercises and Pedagogical Challenges

Deepen your comprehension of macroscopic quantum superpositions and decoherence.

---

### Challenge 1: The Photon Number Parity of Even and Odd Cat States
1. Prove mathematically that for an even cat state $|\psi_+\rangle = \mathcal{N}_+(|\alpha\rangle + |-\alpha\rangle)$, the probability of detecting an odd number of photons is strictly zero: $P_{2m+1} \equiv 0$.
2. For an odd cat state $|\psi_-\rangle = \mathcal{N}_-(|\alpha\rangle - |-\alpha\rangle)$, show that $P_0 \equiv 0$ (the state contains zero vacuum component).

<details>
<summary><b>Click to expand Solution to Challenge 1</b></summary>

**Part 1 Solution:**
The Fock state expansion of a coherent state is:
$$
|\alpha\rangle = e^{-|\alpha|^2/2} \sum_{n=0}^\infty \frac{\alpha^n}{\sqrt{n!}} |n\rangle
$$
For the inverted coherent state:
$$
|-\alpha\rangle = e^{-|\alpha|^2/2} \sum_{n=0}^\infty \frac{(-\alpha)^n}{\sqrt{n!}} |n\rangle = e^{-|\alpha|^2/2} \sum_{n=0}^\infty (-1)^n \frac{\alpha^n}{\sqrt{n!}} |n\rangle
$$
Adding the two states:
$$
|\alpha\rangle + |-\alpha\rangle = e^{-|\alpha|^2/2} \sum_{n=0}^\infty \left[ 1 + (-1)^n \right] \frac{\alpha^n}{\sqrt{n!}} |n\rangle
$$
For odd $n = 2m+1$:
$$
1 + (-1)^{2m+1} = 1 - 1 = 0 \quad \Longrightarrow \quad c_{2m+1} \equiv 0
$$
Therefore, $P_{2m+1} = |c_{2m+1}|^2 \equiv 0$. The state contains exclusively even Fock states!

**Part 2 Solution:**
Subtracting the two states:
$$
|\alpha\rangle - |-\alpha\rangle = e^{-|\alpha|^2/2} \sum_{n=0}^\infty \left[ 1 - (-1)^n \right] \frac{\alpha^n}{\sqrt{n!}} |n\rangle
$$
For $n = 0$:
$$
1 - (-1)^0 = 1 - 1 = 0 \quad \Longrightarrow \quad c_0 \equiv 0
$$
The vacuum probability is strictly zero: $P_0 = 0$.
</details>

---

### Challenge 2: Accelerated Decoherence Scaling
A cavity containing an optical cat state of amplitude $\alpha = 3.0$ experiences a photon dissipation rate $\gamma = 10^3\text{ s}^{-1}$ (cavity lifetime $\tau_{\text{cav}} = 1/\gamma = 1\text{ ms}$).
1. Calculate the mean photon number $\bar{n} \approx |\alpha|^2$.
2. Compute the decoherence time $\tau_{\text{dec}} = 1 / (2\gamma |\alpha|^2)$.
3. Compare $\tau_{\text{dec}}$ to $\tau_{\text{cav}}$.

<details>
<summary><b>Click to expand Solution to Challenge 2</b></summary>

**Solution:**
1. Mean photon number: $\bar{n} \approx |\alpha|^2 = 3.0^2 = 9$ photons.
2. The decoherence rate is:
   $$
   \Gamma_{\text{dec}} = 2\gamma |\alpha|^2 = 2 \times 10^3 \times 9 = 1.8 \times 10^4\text{ s}^{-1}
   $$
   The decoherence time is:
   $$
   \tau_{\text{dec}} = \frac{1}{\Gamma_{\text{dec}}} = \frac{1}{1.8 \times 10^4} \approx 55.6\text{ }\mu\text{s}
   $$
3. Comparison:
   $$
   \frac{\tau_{\text{dec}}}{\tau_{\text{cav}}} = \frac{1}{2|\alpha|^2} = \frac{1}{18} \approx 5.56\%
   $$
   Quantum coherence (the Wigner interference fringes) is destroyed 18 times faster than photons leak out of the cavity!
</details>


## 5. Summary, Reference Table, and Outlook

### Comprehensive Comparison: Cat State Varieties

| Cat State | Superposition Formula | Parity $\langle (-1)^{\hat{n}} \rangle$ | Center Value $W(0, 0)$ | Fringe Phase |
| :--- | :---: | :---: | :---: | :---: |
| **Even Cat $|\psi_+\rangle$** | $\mathcal{N}_+(|\alpha\rangle + |-\alpha\rangle)$ | $+1.0$ | $+\frac{1}{\pi} \approx +0.318$ | $\cos(2\sqrt{2}\alpha p)$ |
| **Odd Cat $|\psi_-\rangle$** | $\mathcal{N}_-(|\alpha\rangle - |-\alpha\rangle)$ | $-1.0$ | $-\frac{1}{\pi} \approx -0.318$ | $-\cos(2\sqrt{2}\alpha p)$ |
| **Yurke-Stoler $|\psi_{\text{YS}}\rangle$** | $\frac{1}{\sqrt{2}}(|\alpha\rangle + i|-\alpha\rangle)$ | $0.0$ | $0.0$ | $\sin(2\sqrt{2}\alpha p)$ |

---

### Key Takeaways
1. **Macroscopic Quantum Interference:** Cat states superpose classical-like coherent states, producing rapid oscillatory interference fringes in phase space with deep negative Wigner valleys.
2. **Sub-Planck Structures:** The momentum period $\Delta p \sim \pi/(\sqrt{2}\alpha)$ scales inversely with separation, unlocking ultra-precise Heisenberg-limited quantum metrology.
3. **Amplified Decoherence:** Environmental loss attenuates off-diagonal quantum coherence at rate $\Gamma_{\text{dec}} = 2\gamma |\alpha|^2$, explaining the rapid classicalization of macroscopic quantum systems.

---

### Primary References
- **Schrödinger, E.** (1935). *Die gegenwärtige Situation in der Quantenmechanik*. Die Naturwissenschaften, 23, 807–812.
- **Yurke, B., & Stoler, D.** (1986). *Generating quantum mechanical superpositions of macroscopically distinguishable states via amplitude dispersion*. Physical Review Letters, 57(1), 13–16.
- **Zurek, W. H.** (2001). *Sub-Planck structure in phase space and its relevance for quantum decoherence*. Nature, 412(6848), 712–717.
- **Haroche, S., & Raimond, J. M.** (2006). *Exploring the Quantum: Atoms, Cavities, and Photons*. Oxford University Press. Chapter 7: Entanglement and Decoherence.

---

### Next in the Curriculum
In **Notebook 2.6: Quantum State Tomography and Wigner Reconstruction**, we conclude Module 2 by simulating balanced homodyne detection and reconstructing the full 2D Wigner function from 1D quadrature histograms using the inverse Radon transform and Maximum Likelihood Estimation (MaxLik).
